In [2]:
import sys
sys.path.append('../utils/')
from utils_models import *
from utils_jax import *

dq.set_device('cpu')
# import warnings
# warnings.filterwarnings("ignore")

In [5]:
solver = dq.solver.Rouchon1(dt = 1e-2)

n_lvls_fluxonium = 20
n_lvls_transmon = 4


Ej_f = 4
Ec_f = 4/5.9
El_f = 4/29.2
qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
    )
g_tf = 0.1
Ec_t = 0.2


def truncate(data: jnp.array):
    return data[:,:]

tot_dim = n_lvls_fluxonium * n_lvls_transmon


In [6]:

def objective(params):    
    sigma = params[0]
    amp_with_2pi = params[1]
    Ej_t = params[2]
    w_d = params[3]
    beta = params[4]
    
    qst = MyTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=10
        )
    

    devices = [qsf, qst]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    tn = qs.promote(qst.ops['n'], t_indx, Ns)

    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
        ])
    system.params["g_tf"] = g_tf
    system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
    driven_op = transform_op_into_dressed_basis_jax(tn, system_evecs_sorted.T)
    # system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
    # w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]


    pulse_length = 6*sigma
    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*jnp.pi),
        'duration': pulse_length,
        'sigma': sigma,
        'beta':beta
    }      

    tlist = jnp.linspace(0,pulse_length,int(pulse_length))

    def _H(t):
        _H =  2 * jnp.pi *truncate(jnp.diag(system_evals_sorted))
        _H += truncate(driven_op) * modified_drag_pulse(t, pulse_shape_args)
        return _H 
    H =  dq.timecallable(_H)
    
    psi0_list = [truncate(dq.basis(tot_dim,find_closest_dressed_index(l*qst.N, product_indices_sorted_by_eval)))
                        for l in [0,1,2]] #00,10,20
    result = dq.sesolve(
                H = H,
                psi0 = psi0_list,
                tsave = tlist,
                solver = solver
                )
        

    rho0 = dq.todm(result.states[0][-1]) / dq.trace(dq.todm(result.states[0][-1])).real
    rho1 = dq.todm(result.states[1][-1]) / dq.trace(dq.todm(result.states[1][-1])).real
    rho2 = dq.todm(result.states[2][-1]) / dq.trace(dq.todm(result.states[2][-1])).real

    f0_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(0*qst.N+1, product_indices_sorted_by_eval)),
                    rho0).real
    
    f1_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(1*qst.N+1, product_indices_sorted_by_eval)),
                    rho1).real
    
    f2_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(2*qst.N+1, product_indices_sorted_by_eval)),
                    rho2).real
    
    return f0_e - f1_e + f2_e

In [8]:
import optax
func = jax.value_and_grad(objective)
params = jnp.array([ 9.70272812e+01,  9.15152087e-03,  3.40891152e+01,  7.17587778e+00,-5.06040151e-04]) # sigma amp_with_2pi  Ej_t  w_d  beta 

optimizer = optax.nadam(learning_rate=jnp.array([1.0,
                                                 1e-3,
                                                 1e-3,
                                                 1e-4,
                                                 1e-3])) 
opt_state = optimizer.init( params )

num_steps = 1000
for step in range(num_steps):
    print(f"iter: {step}, params: {params}")
    val, grads = func(params)
    print(f"\t val={val} grads: {grads}")
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 0, params: [ 9.70272812e+01  9.15152087e-03  3.40891152e+01  7.17587778e+00
 -5.06040151e-04]


TypeError: Solver of type `Rouchon1` is not supported (supported solver types: `Euler`, `Dopri5`, `Dopri8`, `Tsit5`, `Propagator`).